# 06 — Ablations

1. Edge-type removal ablation (RGCN without each auxiliary edge type)
2. Cold-start evaluation (drugs withheld during training)
3. Degree-stratified stability analysis

In [ ]:
import sys
sys.path.insert(0, '..')

import os, json
import torch
import numpy as np
import matplotlib.pyplot as plt

from src.preprocessing import load_processed
from src.graph_builder import build_hetero_data, build_homo_data
from src.models import (
    GCNLinkPredictor, RGCNLinkPredictor,
    RELATION_MAP, NODE_TYPES_ORDER, flatten_hetero_graph
)
from src.training import train_rgcn, train_gcn, sample_negatives
from src.evaluation import compute_link_scores, compute_metrics, compute_ranking_metrics
from src.perturbation import gaussian_noise_perturbation, run_perturbation_trials
from src.stability import full_stability_eval, aggregate_trial_metrics
from src.utils import (
    set_seed, save_checkpoint, save_metrics, load_checkpoint,
    DATA_SPLITS, RESULTS_DIR
)
from torch_geometric.data import HeteroData

set_seed(42)
%matplotlib inline

In [ ]:
SEED = 42
device = torch.device('cpu')

id_maps, edges, stats = load_processed()
num_drugs = len(id_maps['drug'])
hetero_data = build_hetero_data(id_maps, edges)

train_edges = torch.load(os.path.join(DATA_SPLITS, 'train_edges.pt'), weights_only=True)
val_edges = torch.load(os.path.join(DATA_SPLITS, 'val_edges.pt'), weights_only=True)
test_edges = torch.load(os.path.join(DATA_SPLITS, 'test_edges.pt'), weights_only=True)
train_edges_ud = torch.load(os.path.join(DATA_SPLITS, 'train_edges_ud.pt'), weights_only=True)

neg_test = sample_negatives(test_edges, num_drugs, test_edges.shape[1] * 5, seed=888)

## 1. Edge-Type Removal Ablation

Train RGCN variants, each missing one auxiliary edge type.

In [ ]:
EDGE_TYPE_GROUPS = {
    'no_drug_gene': [('drug', 'targets', 'gene'), ('gene', 'targeted_by', 'drug')],
    'no_disease_gene': [('disease', 'associated_with', 'gene'), ('gene', 'associated_with', 'disease')],
    'no_disease_drug': [('disease', 'treated_by', 'drug'), ('drug', 'treats', 'disease')],
}

ablation_results = {}

for ablation_name, excluded_types in EDGE_TYPE_GROUPS.items():
    print(f'\n=== Ablation: {ablation_name} ===')
    set_seed(SEED)
    
    # Build graph without excluded edge types
    abl_hetero = HeteroData()
    for ntype in ['drug', 'gene', 'disease']:
        abl_hetero[ntype].num_nodes = hetero_data[ntype].num_nodes
    abl_hetero['drug', 'interacts', 'drug'].edge_index = train_edges_ud.long()
    for key in hetero_data.edge_types:
        if key != ('drug', 'interacts', 'drug') and key not in excluded_types:
            abl_hetero[key].edge_index = hetero_data[key].edge_index
    
    # Count actual relations in this ablation
    abl_rel_map = {k: v for k, v in RELATION_MAP.items() if k not in excluded_types}
    abl_flat_ei, abl_flat_et, _ = flatten_hetero_graph(abl_hetero, NODE_TYPES_ORDER, abl_rel_map)
    
    # Train
    num_nodes_dict = {ntype: len(id_maps[ntype]) for ntype in NODE_TYPES_ORDER}
    model = RGCNLinkPredictor(
        num_nodes_dict, embed_dim=64,
        num_relations=len(RELATION_MAP), num_bases=2, dropout=0.2
    ).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    _, best_auroc = train_rgcn(
        model, optimizer, train_edges, val_edges, abl_flat_ei, abl_flat_et,
        num_drugs, epochs=300, patience=30, neg_ratio=5, device=device, verbose=False
    )
    
    # Evaluate
    model.eval()
    pos_scores = compute_link_scores(model, test_edges, 'rgcn', abl_flat_ei, abl_flat_et, device)
    neg_scores = compute_link_scores(model, neg_test, 'rgcn', abl_flat_ei, abl_flat_et, device)
    metrics = compute_metrics(pos_scores, neg_scores)
    
    # Stability at sigma=0.10
    with torch.no_grad():
        z_full = model.encode(abl_flat_ei, abl_flat_et)
        z = model.get_drug_embeddings(z_full)
    trials = run_perturbation_trials(z, gaussian_noise_perturbation, [0.10], 10)
    trial_metrics = [full_stability_eval(z, zp) for zp in trials[0.10]]
    stab_agg = aggregate_trial_metrics(trial_metrics)
    metrics['stability_score_01'] = stab_agg['stability_score_mean']
    
    ablation_results[ablation_name] = metrics
    print(f'  AUROC={metrics["auroc"]:.4f}, AUPRC={metrics["auprc"]:.4f}, SS@0.1={metrics["stability_score_01"]:.4f}')
    save_metrics(f'rgcn_{ablation_name}_s{SEED}', metrics)

# Save
with open(os.path.join(RESULTS_DIR, 'ablation_results.json'), 'w') as f:
    json.dump(ablation_results, f, indent=2)
print('\nAblation results saved.')

## 2. Cold-Start Evaluation

In [ ]:
# Load cold-start splits
cold_drugs = torch.load(os.path.join(DATA_SPLITS, 'cold_start_drugs.pt'), weights_only=True)
cold_edges = torch.load(os.path.join(DATA_SPLITS, 'cold_edges.pt'), weights_only=True)
cs_train = torch.load(os.path.join(DATA_SPLITS, 'cs_train_edges.pt'), weights_only=True)
cs_val = torch.load(os.path.join(DATA_SPLITS, 'cs_val_edges.pt'), weights_only=True)

cs_train_ud = torch.cat([cs_train, torch.stack([cs_train[1], cs_train[0]])], dim=1)

cold_drugs_set = set(cold_drugs.tolist())
print(f'Cold-start drugs: {len(cold_drugs_set)}')
print(f'Cold edges: {cold_edges.shape[1]}')
print(f'CS Train: {cs_train.shape[1]}, CS Val: {cs_val.shape[1]}')

In [ ]:
# Train GCN on cold-start split
set_seed(SEED)
cs_gcn = GCNLinkPredictor(num_drugs, embed_dim=64, dropout=0.2).to(device)
cs_gcn_opt = torch.optim.Adam(cs_gcn.parameters(), lr=0.001)
_, cs_gcn_best = train_gcn(
    cs_gcn, cs_gcn_opt, cs_train, cs_val, cs_train_ud, num_drugs,
    epochs=300, patience=30, neg_ratio=5, device=device, verbose=False
)
print(f'CS GCN best val AUROC: {cs_gcn_best:.4f}')

# Train RGCN on cold-start split
set_seed(SEED)
cs_hetero = HeteroData()
for ntype in ['drug', 'gene', 'disease']:
    cs_hetero[ntype].num_nodes = hetero_data[ntype].num_nodes
cs_hetero['drug', 'interacts', 'drug'].edge_index = cs_train_ud.long()
for key in hetero_data.edge_types:
    if key != ('drug', 'interacts', 'drug'):
        cs_hetero[key].edge_index = hetero_data[key].edge_index
cs_flat_ei, cs_flat_et, _ = flatten_hetero_graph(cs_hetero, NODE_TYPES_ORDER, RELATION_MAP)

num_nodes_dict = {ntype: len(id_maps[ntype]) for ntype in NODE_TYPES_ORDER}
cs_rgcn = RGCNLinkPredictor(num_nodes_dict, embed_dim=64, num_relations=len(RELATION_MAP), num_bases=2, dropout=0.2).to(device)
cs_rgcn_opt = torch.optim.Adam(cs_rgcn.parameters(), lr=0.001)
_, cs_rgcn_best = train_rgcn(
    cs_rgcn, cs_rgcn_opt, cs_train, cs_val, cs_flat_ei, cs_flat_et,
    num_drugs, epochs=300, patience=30, neg_ratio=5, device=device, verbose=False
)
print(f'CS RGCN best val AUROC: {cs_rgcn_best:.4f}')

In [ ]:
# Evaluate on cold-start edges
cold_neg = sample_negatives(cold_edges, num_drugs, cold_edges.shape[1] * 5, seed=777)

# GCN cold-start eval
cs_gcn.eval()
pos_s = compute_link_scores(cs_gcn, cold_edges, 'gcn', cs_train_ud, device=device)
neg_s = compute_link_scores(cs_gcn, cold_neg, 'gcn', cs_train_ud, device=device)
cs_gcn_metrics = compute_metrics(pos_s, neg_s)

# GCN stability on cold-start drugs
with torch.no_grad():
    cs_gcn_z = cs_gcn.encode(cs_train_ud)
trials = run_perturbation_trials(cs_gcn_z, gaussian_noise_perturbation, [0.10], 10)
cold_drug_list = sorted(cold_drugs_set)
trial_m = [full_stability_eval(cs_gcn_z, zp) for zp in trials[0.10]]
cs_gcn_stab = aggregate_trial_metrics(trial_m)
cs_gcn_metrics['stability_score_01'] = cs_gcn_stab['stability_score_mean']

# RGCN cold-start eval
cs_rgcn.eval()
pos_s = compute_link_scores(cs_rgcn, cold_edges, 'rgcn', cs_flat_ei, cs_flat_et, device)
neg_s = compute_link_scores(cs_rgcn, cold_neg, 'rgcn', cs_flat_ei, cs_flat_et, device)
cs_rgcn_metrics = compute_metrics(pos_s, neg_s)

with torch.no_grad():
    cs_rgcn_z_full = cs_rgcn.encode(cs_flat_ei, cs_flat_et)
    cs_rgcn_z = cs_rgcn.get_drug_embeddings(cs_rgcn_z_full)
trials = run_perturbation_trials(cs_rgcn_z, gaussian_noise_perturbation, [0.10], 10)
trial_m = [full_stability_eval(cs_rgcn_z, zp) for zp in trials[0.10]]
cs_rgcn_stab = aggregate_trial_metrics(trial_m)
cs_rgcn_metrics['stability_score_01'] = cs_rgcn_stab['stability_score_mean']

print(f"{'Metric':<20} {'GCN Cold':>12} {'RGCN Cold':>12}")
print('-' * 46)
for m in ['auroc', 'auprc', 'stability_score_01']:
    print(f"{m:<20} {cs_gcn_metrics.get(m,0):>12.4f} {cs_rgcn_metrics.get(m,0):>12.4f}")

coldstart_data = {'gcn_cold': cs_gcn_metrics, 'rgcn_cold': cs_rgcn_metrics}
with open(os.path.join(RESULTS_DIR, 'coldstart_results.json'), 'w') as f:
    json.dump(coldstart_data, f, indent=2)
print('\nCold-start results saved.')